# Project 1 — Sales & Revenue Dashboard
### The Flying Enterprise Cork | Python · Pandas · Matplotlib

**Author:** Bhunesh Dadheech | MSc Data Science & Analytics, UCC Cork  
**Role context:** This notebook replicates the executive dashboard work done at Codeup India, where 8 Tableau dashboards were built and adopted by leadership for weekly business reviews.

---
## What this notebook does
1. Loads and validates 163,742 pub transactions (2023–2024)
2. Engineers headline KPIs — revenue, gross profit, GP margin, YoY growth
3. Analyses patterns by hour, day, month, category, and event days
4. Renders a multi-panel executive dashboard


In [1]:
# ── CELL 1: Imports ──────────────────────────────────────────────────────────
# We keep imports minimal and purposeful — interviewers notice bloated import blocks.
# Pandas for all data manipulation, NumPy for numerical ops, Matplotlib for charts.

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')        # Use non-interactive backend (safe for all environments)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import warnings, os

warnings.filterwarnings('ignore')  # Suppress Pandas version warnings for clean output
pd.set_option('display.float_format', '{:,.2f}'.format)  # Consistent number formatting
print("✅ Imports loaded")

✅ Imports loaded


## Step 1 — Load & Validate Data

**Why validate first?**  
Never skip this step — analysis built on dirty data produces wrong insights.
The goal is to catch nulls, duplicates, and out-of-range values *before* any calculation.


In [2]:
# ── CELL 2: Load data ────────────────────────────────────────────────────────
# parse_dates converts the date column from string to datetime automatically.
# This is more efficient than calling pd.to_datetime() separately after loading.

DATA_PATH = 'data'
df = pd.read_csv("C:/Users/Bhunesh/Flying Enterprise Projects/data/transactions.csv", parse_dates=['timestamp', 'date'])
df['date'] = pd.to_datetime(df['date'])

print(f"Shape:        {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range:   {df['date'].min().date()} → {df['date'].max().date()}")
print(f"\nColumn dtypes:")
print(df.dtypes)

Shape:        163,742 rows × 20 columns
Date range:   2023-01-01 → 2024-12-31

Column dtypes:
transaction_id               str
timestamp         datetime64[us]
date              datetime64[us]
hour                       int64
day_of_week                  str
week                       int64
month                      int64
month_name                   str
year                       int64
product                      str
category                     str
quantity                   int64
unit_price               float64
revenue                  float64
cost                     float64
gross_profit             float64
staff_id                     str
staff_name                   str
is_event_day                bool
is_weekend                  bool
dtype: object


In [3]:
# ── CELL 3: Data quality checks ──────────────────────────────────────────────
# In production I run these as assertions so the pipeline fails loudly on bad data.
# Here we report them for visibility, then confirm we're clean before proceeding.

print("=== DATA QUALITY REPORT ===")
print(f"Total rows:          {len(df):,}")
print(f"Null values:         {df.isnull().sum().sum()}")
print(f"Duplicate txn IDs:   {df.duplicated('transaction_id').sum()}")
print(f"Negative revenues:   {(df['revenue'] < 0).sum()}")
print(f"Zero quantities:     {(df['quantity'] == 0).sum()}")
print(f"\nRevenue stats:")
print(df['revenue'].describe())

# Confirm we're clean before moving on
assert df['transaction_id'].duplicated().sum() == 0, "Duplicate transaction IDs found!"
assert df['revenue'].min() > 0, "Negative or zero revenue found!"
print("\n✅ All quality checks passed")

=== DATA QUALITY REPORT ===
Total rows:          163,742
Null values:         0
Duplicate txn IDs:   0
Negative revenues:   0
Zero quantities:     0

Revenue stats:
count   163,742.00
mean         12.56
std           7.78
min           3.50
25%           7.50
50%          10.50
75%          15.50
max          46.50
Name: revenue, dtype: float64

✅ All quality checks passed


## Step 2 — Core KPI Calculation

**Design decision:** Calculate all KPIs from the raw transaction grain, not from aggregated summaries.  
This prevents compounding rounding errors and ensures every KPI is traceable back to source data.


In [4]:
# ── CELL 4: Headline KPIs ────────────────────────────────────────────────────
# These 7 numbers are what an exec sees first on a dashboard.
# Each is a single aggregation — no intermediate tables needed.

total_revenue      = df['revenue'].sum()
total_cost         = df['cost'].sum()
total_gp           = df['gross_profit'].sum()
gp_margin          = (total_gp / total_revenue) * 100
avg_transaction    = df['revenue'].mean()
avg_daily_revenue  = df.groupby('date')['revenue'].sum().mean()
total_transactions = len(df)

# YoY: compare total 2024 vs total 2023
rev_by_year = df.groupby('year')['revenue'].sum()
yoy_growth  = ((rev_by_year[2024] - rev_by_year[2023]) / rev_by_year[2023]) * 100

print("=" * 50)
print("  FLYING ENTERPRISE — HEADLINE KPIs")
print("=" * 50)
print(f"  Total Revenue (2yr):    €{total_revenue:>12,.2f}")
print(f"  Total Cost:             €{total_cost:>12,.2f}")
print(f"  Gross Profit:           €{total_gp:>12,.2f}")
print(f"  GP Margin:              {gp_margin:>11.1f}%")
print(f"  Avg Transaction:        €{avg_transaction:>11.2f}")
print(f"  Avg Daily Revenue:      €{avg_daily_revenue:>11.2f}")
print(f"  Total Transactions:     {total_transactions:>12,}")
print(f"  YoY Revenue Growth:     {yoy_growth:>+11.1f}%")

  FLYING ENTERPRISE — HEADLINE KPIs
  Total Revenue (2yr):    €2,056,923.70
  Total Cost:             €  609,916.80
  Gross Profit:           €1,447,006.90
  GP Margin:                     70.3%
  Avg Transaction:        €      12.56
  Avg Daily Revenue:      €    2813.85
  Total Transactions:          163,742
  YoY Revenue Growth:            -0.2%


## Step 3 — Time-Based Analysis

**Insight pattern:** Understanding *when* customers spend reveals scheduling and operational opportunities.  
- **Hourly analysis** → optimal opening hours, staffing peaks  
- **Day-of-week analysis** → weekend vs weekday stock ordering  
- **Monthly analysis** → seasonal promotions, inventory planning  


In [5]:
# ── CELL 5: Monthly trend ────────────────────────────────────────────────────
# Group by year + month to build the 24-month trend.
# We include month_name for readable labels and sort by year then month numerically.

monthly = (df.groupby(['year', 'month', 'month_name'])
             .agg(
                 revenue=('revenue', 'sum'),
                 transactions=('transaction_id', 'count'),
                 gp=('gross_profit', 'sum')
             )
             .reset_index()
             .sort_values(['year', 'month']))

monthly['gp_margin'] = (monthly['gp'] / monthly['revenue']) * 100
monthly['label']     = monthly['month_name'].str[:3] + ' ' + monthly['year'].astype(str).str[-2:]

print("Monthly revenue summary (first 6 months):")
print(monthly[['label', 'revenue', 'gp_margin']].head(6).to_string(index=False))

Monthly revenue summary (first 6 months):
 label    revenue  gp_margin
Jan 23  60,290.80      70.35
Feb 23  57,096.00      70.41
Mar 23  86,919.00      70.37
Apr 23  81,897.00      70.32
May 23  79,552.40      70.28
Jun 23 115,352.60      70.30


In [6]:
# ── CELL 6: Hourly + day-of-week patterns ────────────────────────────────────
# Average revenue per transaction by hour tells us *when* people spend most per visit.
# Total revenue by day tells us *which days* are busiest in absolute terms.

hourly = df.groupby('hour')['revenue'].mean().reset_index()
hourly.columns = ['hour', 'avg_revenue']

DOW_ORDER = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily_dow = (df.groupby('day_of_week')['revenue']
               .sum()
               .reindex(DOW_ORDER)
               .reset_index())
daily_dow.columns = ['day', 'revenue']

peak_hour = hourly.loc[hourly['avg_revenue'].idxmax(), 'hour']
peak_day  = daily_dow.loc[daily_dow['revenue'].idxmax(), 'day']

print(f"Peak trading hour: {peak_hour}:00")
print(f"Peak revenue day:  {peak_day}")
print(f"\nTop 3 hours by avg revenue:")
print(hourly.nlargest(3, 'avg_revenue').to_string(index=False))

Peak trading hour: 12:00
Peak revenue day:  Saturday

Top 3 hours by avg revenue:
 hour  avg_revenue
   12        12.76
   15        12.70
   18        12.62


In [7]:
# ── CELL 7: Category analysis ────────────────────────────────────────────────
# We calculate revenue share as a % of total — interviewers often ask how you'd do this.
# The subquery approach (dividing by total_revenue) avoids a self-join.

cat = (df.groupby('category')
         .agg(
             revenue=('revenue', 'sum'),
             transactions=('transaction_id', 'count'),
             gp=('gross_profit', 'sum'),
         )
         .sort_values('revenue', ascending=False)
         .reset_index())

cat['revenue_share'] = (cat['revenue'] / total_revenue * 100).round(1)
cat['gp_margin']     = (cat['gp'] / cat['revenue'] * 100).round(1)

print("Revenue by category:")
print(cat[['category', 'revenue', 'revenue_share', 'gp_margin']].to_string(index=False))

Revenue by category:
     category    revenue  revenue_share  gp_margin
         Food 884,934.50          43.00      65.90
 Draught Beer 431,268.30          21.00      73.30
      Spirits 329,953.50          16.00      73.50
         Wine 311,029.00          15.10      71.70
Non-Alcoholic  99,738.40           4.80      82.20


In [8]:
# ── CELL 8: Event day analysis ────────────────────────────────────────────────
# We compare average daily revenue on event days vs normal days.
# Using mean (not sum) accounts for the different number of event vs non-event days.

event_daily = df.groupby(['date', 'is_event_day'])['revenue'].sum().reset_index()
event_agg   = event_daily.groupby('is_event_day')['revenue'].mean()

event_uplift = (event_agg[True] / event_agg[False] - 1) * 100
print(f"Avg daily revenue — normal days: €{event_agg[False]:,.2f}")
print(f"Avg daily revenue — event days:  €{event_agg[True]:,.2f}")
print(f"Event day uplift:                {event_uplift:+.0f}%")

Avg daily revenue — normal days: €2,743.66
Avg daily revenue — event days:  €7,408.08
Event day uplift:                +170%


## Step 4 — Executive Dashboard

**Design decisions:**
- Dark background (#1a1a2e) — professional, high contrast, matches modern BI tool aesthetics
- GridSpec layout — precise control over subplot sizing (better than `plt.subplots()` for dashboards)
- Colour-coded by year — instant visual distinction between 2023 and 2024
- All KPIs in top row — executives see the headline numbers before any charts


In [9]:
# ── CELL 9: Build dashboard ───────────────────────────────────────────────────
# Full 5-panel executive dashboard saved as PNG.
# Run sales_dashboard.py from the scripts/ folder for the complete version.

DARK='#1a1a2e'; MID='#16213e'; ACCENT='#e94560'
GOLD='#f5a623'; GREEN='#00d4aa'; BLUE='#4a9eff'; WHITE='#f0f0f0'; MUTED='#8888aa'

fig = plt.figure(figsize=(20, 14), facecolor=DARK)
fig.suptitle('THE FLYING ENTERPRISE CORK — SALES ANALYTICS DASHBOARD 2023–2024',
             fontsize=16, color=WHITE, fontweight='bold', y=0.98)

gs = GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35,
              top=0.93, bottom=0.06, left=0.06, right=0.97)

# KPI cards
kpi_data = [
    ('Total Revenue',  f'€{total_revenue/1e6:.2f}M', GOLD),
    ('Gross Profit',   f'€{total_gp/1e6:.2f}M',      GREEN),
    ('GP Margin',      f'{gp_margin:.1f}%',           BLUE),
    ('YoY Growth',     f'{yoy_growth:+.1f}%',         ACCENT),
]
for i, (label, value, color) in enumerate(kpi_data):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(MID); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.text(0.5, 0.72, value, transform=ax.transAxes,
            ha='center', fontsize=26, fontweight='bold', color=color)
    ax.text(0.5, 0.28, label, transform=ax.transAxes,
            ha='center', fontsize=11, color=MUTED)

# Monthly trend
ax2 = fig.add_subplot(gs[1, :3])
ax2.set_facecolor(MID)
colors_bar = [BLUE if y == 2023 else GOLD for y in monthly['year']]
ax2.bar(range(len(monthly)), monthly['revenue']/1000, color=colors_bar, alpha=0.85, width=0.7)
ax2.plot(range(len(monthly)), monthly['revenue']/1000, color=GREEN, lw=2, marker='o', ms=4)
ax2.set_xticks(range(len(monthly)))
ax2.set_xticklabels(monthly['label'], rotation=45, ha='right', fontsize=8, color=MUTED)
ax2.set_title('Monthly Revenue Trend', color=WHITE, fontsize=12, pad=8)
ax2.set_facecolor(MID); ax2.tick_params(colors=MUTED)
for sp in ax2.spines.values(): sp.set_color('#333355')
ax2.grid(axis='y', color='#333355', alpha=0.5)

# Category pie
ax3 = fig.add_subplot(gs[1, 3])
ax3.set_facecolor(MID)
ax3.pie(cat['revenue'], labels=cat['category'], autopct='%1.1f%%',
        colors=[GOLD,BLUE,GREEN,ACCENT,'#aa44ff'][:len(cat)],
        pctdistance=0.78, startangle=140,
        textprops={'fontsize':7.5,'color':WHITE},
        wedgeprops={'linewidth':1.5,'edgecolor':DARK})
ax3.set_title('Revenue by Category', color=WHITE, fontsize=12, pad=8)

# Hourly
ax4 = fig.add_subplot(gs[2, :2])
ax4.set_facecolor(MID)
bar_colors = [ACCENT if h == peak_hour else BLUE for h in hourly['hour']]
ax4.bar(hourly['hour'], hourly['avg_revenue'], color=bar_colors, alpha=0.85, width=0.7)
ax4.set_title('Avg Revenue by Hour', color=WHITE, fontsize=12, pad=8)
ax4.tick_params(colors=MUTED)
for sp in ax4.spines.values(): sp.set_color('#333355')
ax4.grid(axis='y', color='#333355', alpha=0.5)

# Day of week
ax5 = fig.add_subplot(gs[2, 2:])
ax5.set_facecolor(MID)
dow_colors = [ACCENT if d == peak_day else GREEN for d in daily_dow['day']]
ax5.bar(range(7), daily_dow['revenue']/1000, color=dow_colors, alpha=0.85, width=0.6)
ax5.set_xticks(range(7))
ax5.set_xticklabels([d[:3] for d in daily_dow['day']], color=MUTED, fontsize=9)
ax5.set_title('Revenue by Day of Week', color=WHITE, fontsize=12, pad=8)
ax5.tick_params(colors=MUTED)
for sp in ax5.spines.values(): sp.set_color('#333355')
ax5.grid(axis='y', color='#333355', alpha=0.5)

out = os.path.join('..', 'outputs')
os.makedirs(out, exist_ok=True)
plt.savefig("C:/Users/Bhunesh/Flying Enterprise Projects/project1_sales_dashboard.png", dpi=150, bbox_inches='tight', facecolor=DARK)
print("✅ Dashboard saved → outputs/project1_sales_dashboard.png")
plt.close()

✅ Dashboard saved → outputs/project1_sales_dashboard.png


## Summary & Interview Talking Points

**What this project shows:**
- End-to-end data workflow from raw CSV to executive-ready output
- Strong KPI engineering — not just `df.groupby().sum()`, but margin calculations, YoY growth, event uplift
- Professional visualisation choices (GridSpec, colour theory, data-ink ratio)
- Data validation before analysis — a production habit many junior analysts skip

**Likely interview questions:**
- *"How would you schedule this to run daily automatically?"* → Apache Airflow DAG (see Project 2)
- *"How would you put this in Tableau instead?"* → Export aggregated CSVs, connect as data source, replicate layout
- *"How did you handle the YoY calculation?"* → `groupby('year')['revenue'].sum()` then percentage change
